<a href="https://colab.research.google.com/github/raymondsum2002-wq/HongKongJockeyClub_HorseRacePrediction/blob/main/%E9%A6%99%E6%B8%AF%E8%B3%BD%E9%A6%AC%E9%A0%90%E6%B8%AC%E7%B3%BB%E7%B5%B1v11_3L.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 程式用途: 香港賽馬五大子模型機器學習預測架構 (v11.3 子模型解析與自動報表升級版)
# 版本號碼: v11.3_Production
# 改進與修正重點:
#   1. 報表分頁化：不再混於同一頁，每場賽事自動產生獨立的 Excel 分頁 (Sheet)。
#   2. 子模型透明化：於報表中獨立列出 S1~S5 各子模型之評分，利於賽後覆盤與權重調整。
#   3. 視覺化五彩標籤：針對每個子模型分數前 5 名的馬匹，依序加上 [橙, 黃, 綠, 藍, 紫] 底色。
#   4. 自動目錄管理：自動建立 /MyDrive/Racing_Report/ 資料夾，並規範檔案命名格式。
#   5. 核心防護延續：保留日期解析修復 (Date Swap Bug Fix) 與強制馬號對接賠率之邏輯。
# ==============================================================================

import glob
import os
import re
import sys
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from openpyxl.styles import PatternFill
from datetime import datetime, timedelta

# 忽略非關鍵系統提示，保持控制台輸出乾淨清爽
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import drive
drive.mount('/content/drive')

# ==============================================================================
# 🎛️ Colab 互動式選單介面 (請於點選右側 Form 介面調整)
# ==============================================================================
#@markdown ### 📌 1. 預測模式與賽事日期設定
RUN_MODE = "Live_Racecard" #@param ["Live_Racecard", "Historical_Backtest"]
AUTO_TODAY_DATE = False #@param {type:"boolean"}
PREDICT_DATE = "2026-09-16" #@param {type:"date"}

if AUTO_TODAY_DATE:
    # 自動抓取系統當下時間，並加上 8 小時轉換為香港時區 (UTC+8) 的當天日期
    PREDICT_DATE = (datetime.utcnow() + timedelta(hours=8)).strftime('%Y-%m-%d')

#@markdown ### ⚙️ 2. 演算法進階參數設定
TEMPERATURE_TAU = 1.5 #@param {type:"number"}
# 預設儲存目錄改為 Racing_Report
EXCEL_OUTPUT_DIR = "/content/drive/MyDrive/Racing_Report" #@param {type:"string"}

print(f"=== 🚀 香港賽馬五大子模型系統 [Version: v11.3_Production] ===")
print(f"當前執行模式: {RUN_MODE}")
if RUN_MODE == "Live_Racecard":
    print(f"指定預測日期: {PREDICT_DATE}")

def find_file_fuzzy(pattern_keyword, root_dir='/content/drive/MyDrive'):
    files = glob.glob(f'{root_dir}/**/*{pattern_keyword}*', recursive=True)
    if not files:
        files = glob.glob(f'***{pattern_keyword}*', recursive=True)
    real_files = [f for f in files if not f.endswith('.gsheet')]
    return real_files[0] if real_files else (files[0] if files else None)

def load_data_file(file_path):
    if not file_path or not os.path.exists(file_path):
        return pd.DataFrame()
    if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        df = pd.read_excel(file_path)
    else:
        try:
            df = pd.read_csv(file_path, encoding='utf-8-sig', low_memory=False)
        except:
            try:
                df = pd.read_csv(file_path, encoding='gbk', low_memory=False)
            except:
                df = pd.read_csv(file_path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    return df

def normalize_date_str(date_val):
    if pd.isna(date_val):
        return ""
    s = str(date_val).strip()
    if not s:
        return ""

    clean_digits = re.sub(r'\D', '', s)
    if len(clean_digits) == 8:
        return f"{clean_digits[:4]}-{clean_digits[4:6]}-{clean_digits[6:]}"

    if re.match(r'^\d{4}[-/.]\d{1,2}[-/.]\d{1,2}', s):
        dt = pd.to_datetime(s, errors='coerce')
        if pd.notna(dt):
            return dt.strftime('%Y-%m-%d')

    if re.match(r'^\d{1,2}[-/.]\d{1,2}[-/.]\d{4}', s):
        dt = pd.to_datetime(s, dayfirst=True, errors='coerce')
        if pd.notna(dt):
            return dt.strftime('%Y-%m-%d')

    dt = pd.to_datetime(s, errors='coerce')
    if pd.notna(dt):
        return dt.strftime('%Y-%m-%d')

    return s

def clean_race_num(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    nums = re.findall(r'\d+', s)
    if nums:
        return int(nums[-1])
    return np.nan

def clean_horse_name(name_str):
    if pd.isna(name_str):
        return ""
    s = str(name_str).strip()
    s = re.sub(r'^(nan|NaN|NAN|\s+)', '', s).strip()
    s = re.sub(r'\([A-Z0-9]+\)', '', s).strip()
    s = re.sub(r'\s+', '', s)
    return s

def clean_horse_no(row):
    for col in ['馬號', '排位馬號', 'No.', 'NO.', 'No', 'no', '馬號/No.', '編號']:
        if col in row and pd.notna(row[col]):
            val = str(row[col]).replace('.0', '').strip()
            if val and val.lower() != 'nan':
                return val
    return "-"

def extract_odds_columns(df):
    win_col, pla_col = None, None
    win_candidates = ['獨贏賠率', '獨贏', '賠率', 'WIN', 'Win', 'win', 'WIN_ODDS', 'Odds', 'ODDS', '獨贏 (WIN)', 'Latest_Odds']
    for c in win_candidates:
        if c in df.columns:
            win_col = c
            break
    if not win_col:
        for col in df.columns:
            if any(k in str(col).lower() for k in ['win', '獨贏']) and 'place' not in str(col).lower():
                win_col = col
                break

    pla_candidates = ['位置賠率', '位置', 'PLA', 'Pla', 'pla', 'PLA_ODDS', 'Place', 'PLACE', '位置 (PLA)']
    for c in pla_candidates:
        if c in df.columns:
            pla_col = c
            break
    if not pla_col:
        for col in df.columns:
            if any(k in str(col).lower() for k in ['pla', '位置', 'place']):
                pla_col = col
                break

    return win_col, pla_col

path_v7 = find_file_fuzzy('HK_Racing_V7_6_Archive_Two_Hos')
path_st = find_file_fuzzy('ST sectional time')
path_hv = find_file_fuzzy('HV sectional time')

print('\n=== 📁 數據檔案讀取對接確認 ===')
print('主特徵檔 (V7_6):', path_v7)
print('沙田標準時間檔:', path_st)
print('快活谷標準時間檔:', path_hv)

df_v7 = load_data_file(path_v7)
df_v7['日期_dt'] = pd.to_datetime(df_v7['日期'].apply(normalize_date_str), errors='coerce')
df_v7['真實名次_num'] = pd.to_numeric(df_v7['名次'], errors='coerce')
if '馬名' in df_v7.columns:
    df_v7['馬名_clean'] = df_v7['馬名'].apply(clean_horse_name)

df_racecard = pd.DataFrame()
target_date_norm = normalize_date_str(PREDICT_DATE)
target_date_clean_digits = re.sub(r'\D', '', target_date_norm)

if RUN_MODE == "Live_Racecard":
    racecard_path = find_file_fuzzy(f'Racecard_Stable_{target_date_clean_digits}')
    if not racecard_path:
        racecard_path = find_file_fuzzy(f'{target_date_clean_digits}')

    print(f'指定日期排位檔路徑:', racecard_path)
    if racecard_path and os.path.exists(racecard_path):
        df_racecard = load_data_file(racecard_path)
        if '馬名' in df_racecard.columns:
            df_racecard['馬名_clean'] = df_racecard['馬名'].apply(clean_horse_name)
        print(f'✅ 成功載入排位檔，共 {len(df_racecard)} 匹次參賽馬匹。')
    else:
        print(f'\n❌【嚴重錯誤】未找到日期 [{PREDICT_DATE}] (精準格式 {target_date_clean_digits}) 之排位檔案！')
        print(f'💡 請檢查 Google Drive 中是否已放上命名如 Racecard_Stable_{target_date_clean_digits}.csv 的排位檔。')

    odds_path = find_file_fuzzy('HKJC_Historical_Data', root_dir='/content/drive/MyDrive/HKJC_Human_Report')
    if not odds_path:
        odds_path = find_file_fuzzy('HKJC_Historical_Data')

    print('HKJC 動態賠率檔路徑:', odds_path)

    odds_dict_win_race = {}
    odds_dict_win_horse = {}
    odds_dict_pla_race = {}
    odds_dict_pla_horse = {}

    if odds_path and os.path.exists(odds_path):
        df_odds = load_data_file(odds_path)
        print(f'✅ 成功載入動態賠率數據，共 {len(df_odds)} 條紀錄。')

        for date_col in ['日期', 'Date', 'date', 'RaceDate', 'RACE_DATE', '賽事日期']:
            if date_col in df_odds.columns:
                df_odds['Date_norm'] = df_odds[date_col].apply(normalize_date_str)
                df_odds_today = df_odds[df_odds['Date_norm'] == target_date_norm]
                if not df_odds_today.empty:
                    df_odds = df_odds_today
                    print(f'📅 已篩選 {target_date_norm} 當日賠率數據，共 {len(df_odds)} 條。')
                else:
                    print(f'⚠️ 日期篩選: 未能在賠率檔中找到日期 [{target_date_norm}]，嘗試全量模糊匹配...')
                break

        win_col_name, pla_col_name = extract_odds_columns(df_odds)
        print(f'🔍 識別彩池賠率欄位名稱 -> 獨贏 (WIN): [{win_col_name}] | 位置 (PLA): [{pla_col_name}]')

        if win_col_name:
            df_odds['馬號_clean'] = df_odds.apply(clean_horse_no, axis=1)
            df_odds['Latest_WIN'] = pd.to_numeric(df_odds[win_col_name], errors='coerce')

            if pla_col_name:
                df_odds['Latest_PLA'] = pd.to_numeric(df_odds[pla_col_name], errors='coerce')
            else:
                df_odds['Latest_PLA'] = np.nan

            df_odds_valid = df_odds.dropna(subset=['Latest_WIN']).copy()

            if '場次' in df_odds_valid.columns:
                df_odds_valid['場次_num'] = df_odds_valid['場次'].apply(clean_race_num)
                odds_dict_win_race = df_odds_valid.groupby(['場次_num', '馬號_clean'])['Latest_WIN'].last().to_dict()
                odds_dict_pla_race = df_odds_valid.groupby(['場次_num', '馬號_clean'])['Latest_PLA'].last().to_dict()

            odds_dict_win_horse = df_odds_valid.groupby('馬號_clean')['Latest_WIN'].last().to_dict()
            odds_dict_pla_horse = df_odds_valid.groupby('馬號_clean')['Latest_PLA'].last().to_dict()

    if not df_racecard.empty:
        df_racecard['場次_num'] = df_racecard.get('場次', 1).apply(clean_race_num)
        if '馬名_clean' not in df_racecard.columns:
            df_racecard['馬名_clean'] = df_racecard.get('馬名', '').apply(clean_horse_name)
        if '馬號_clean' not in df_racecard.columns:
            df_racecard['馬號_clean'] = df_racecard.apply(clean_horse_no, axis=1)

        native_win_col, native_pla_col = extract_odds_columns(df_racecard)
        if native_win_col:
            df_racecard['Native_WIN'] = pd.to_numeric(df_racecard[native_win_col], errors='coerce')
        else:
            df_racecard['Native_WIN'] = np.nan

        if native_pla_col:
            df_racecard['Native_PLA'] = pd.to_numeric(df_racecard[native_pla_col], errors='coerce')
        else:
            df_racecard['Native_PLA'] = np.nan

        def resolve_win_odds(row):
            r_num = row.get('場次_num', np.nan)
            h_no = row.get('馬號_clean', '')
            if (r_num, h_no) in odds_dict_win_race:
                return odds_dict_win_race[(r_num, h_no)]
            if h_no in odds_dict_win_horse:
                return odds_dict_win_horse[h_no]
            if pd.notna(row.get('Native_WIN')):
                return row.get('Native_WIN')
            return np.nan

        def resolve_pla_odds(row):
            r_num = row.get('場次_num', np.nan)
            h_no = row.get('馬號_clean', '')
            if (r_num, h_no) in odds_dict_pla_race:
                return odds_dict_pla_race[(r_num, h_no)]
            if h_no in odds_dict_pla_horse:
                return odds_dict_pla_horse[h_no]
            if pd.notna(row.get('Native_PLA')):
                return row.get('Native_PLA')
            return np.nan

        df_racecard['獨贏賠率'] = df_racecard.apply(resolve_win_odds, axis=1)
        df_racecard['位置賠率'] = df_racecard.apply(resolve_pla_odds, axis=1)

        matched_count = df_racecard['獨贏賠率'].notna().sum()
        print(f'🔄 賠率對接結果: 成功對接 {matched_count} / {len(df_racecard)} 匹馬匹之即時獨贏賠率！')

def parse_time_robust(t_str):
    if pd.isna(t_str) or not str(t_str).strip():
        return np.nan
    s = str(t_str).strip().replace(':', '.')
    parts = s.split('.')
    if len(parts) == 3:
        return float(parts[0]) * 60 + float(parts[1]) + float(parts[2]) / 100.0
    elif len(parts) == 2:
        return (
            float(parts[0]) * 60 + float(parts[1])
            if float(parts[0]) > 2
            else float(parts[0]) + float(parts[1]) / 100.0
        )
    try:
        return float(s)
    except:
        return np.nan

df_v7['Race_Time_Sec'] = df_v7['完成時間'].apply(parse_time_robust)
df_v7['Min_Race_Time'] = df_v7.groupby(['日期', '場次'])['Race_Time_Sec'].transform('min')
df_v7['Time_Gap_Sec'] = (df_v7['Race_Time_Sec'] - df_v7['Min_Race_Time']).fillna(2.0)
df_v7['Target_Relevance'] = 10.0 * np.exp(-2.0 * df_v7['Time_Gap_Sec'])

if RUN_MODE == "Live_Racecard" and not df_racecard.empty:
    df_racecard['日期_dt'] = pd.to_datetime(target_date_norm, errors='coerce')
    df_racecard['真實名次_num'] = np.nan
    df_racecard['Target_Relevance'] = np.nan
    df_racecard['Time_Gap_Sec'] = np.nan
    df_racecard['Is_Racecard'] = 1
    df_v7['Is_Racecard'] = 0
    df_combined = pd.concat([df_v7, df_racecard], ignore_index=True)
else:
    df_v7['Is_Racecard'] = 0
    df_combined = df_v7.copy()

if '馬名_clean' not in df_combined.columns:
    df_combined['馬名_clean'] = df_combined['馬名'].apply(clean_horse_name)

# === 新增：清理騎師名字括號並提取讓磅數字 ===
def extract_jockey_info(j_str):
    if pd.isna(j_str):
        return pd.Series(['-', 0.0])
    s = str(j_str).strip()
    match = re.search(r'\(([-+]?\d+)\)', s)
    allowance = 0.0
    if match:
        # 取得讓磅數值 (例如從 "-2" 取得 -2.0)
        allowance = float(match.group(1))
    # 清除括號及其內的數字，還原純名字
    clean_name = re.sub(r'\s*\([^)]*\)', '', s).strip()
    return pd.Series([clean_name, allowance])

if '騎師' in df_combined.columns:
    df_combined[['騎師', '讓磅_num']] = df_combined['騎師'].apply(extract_jockey_info)
else:
    df_combined['騎師'] = '-'
    df_combined['讓磅_num'] = 0.0

df_combined = df_combined.sort_values(by=['馬名_clean', '日期_dt']).reset_index(drop=True)

for sec_idx in [1, 2, 3]:
    col_name = f'拆分200m_{sec_idx}'
    df_combined[f'Sec_{sec_idx}_Val'] = pd.to_numeric(df_combined.get(col_name, 0), errors='coerce').fillna(0)
    df_combined[f'Hist_Sec_{sec_idx}'] = df_combined.groupby('馬名_clean')[f'Sec_{sec_idx}_Val'].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    ).fillna(0)

df_combined['Hist_Sec1_2_Delta'] = df_combined['Hist_Sec_1'] + df_combined['Hist_Sec_2']

df_combined['Is_Top3'] = df_combined['真實名次_num'].isin([1, 2, 3]).astype(int)
df_combined['Race_Strength_Score'] = df_combined.groupby('馬名_clean')['Is_Top3'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
).fillna(0.20)

df_combined['主持A_num'] = pd.to_numeric(df_combined.get('主持A狀態', 0), errors='coerce').fillna(0)
df_combined['主持B_num'] = pd.to_numeric(df_combined.get('主持B狀態', 0), errors='coerce').fillna(0)
df_combined['Host_Avg_Score'] = (df_combined['主持A_num'] + df_combined['主持B_num']) / 2.0
if 'Gemini_Score' in df_combined.columns:
    df_combined['Gemini_Workload_Score'] = pd.to_numeric(df_combined['Gemini_Score'], errors='coerce').fillna(3.0)
else:
    df_combined['Gemini_Workload_Score'] = 3.0
df_combined['Host_Double_Low_Reject'] = ((df_combined['主持A_num'] == 1) & (df_combined['主持B_num'] == 1)).astype(int)

df_combined['Habitual_Pace_Type'] = np.where(
    df_combined['Hist_Sec1_2_Delta'] > 0.5, 1,
    np.where(df_combined['Hist_Sec1_2_Delta'] > 0.2, 2,
    np.where(df_combined['Hist_Sec1_2_Delta'] > -0.1, 3,
    np.where(df_combined['Hist_Sec1_2_Delta'] > -0.4, 4, 5)))
)

df_combined['場次_num'] = df_combined.get('場次', 1).apply(clean_race_num)

group_sizes = df_combined.groupby(['日期', '場次_num'])['Hist_Sec1_2_Delta'].transform('count')
ranks_in_race = df_combined.groupby(['日期', '場次_num'])['Hist_Sec1_2_Delta'].rank(method='first', ascending=False)
q_calculated = np.ceil((ranks_in_race / (group_sizes + 1e-9)) * 5).astype(int)
q_calculated = np.clip(q_calculated, 1, 5)

df_combined['Relative_Pace_Role_Quintile'] = np.where(group_sizes < 5, 3, q_calculated)
df_combined['Pace_Role_Shift'] = df_combined['Relative_Pace_Role_Quintile'] - df_combined['Habitual_Pace_Type']

df_combined['is_eff_front'] = ((df_combined['Hist_Sec1_2_Delta'] > 0.2) & (df_combined['Host_Double_Low_Reject'] == 0)).astype(int)
df_combined['is_eff_closer'] = ((df_combined['Hist_Sec_3'] > 0.2) & (df_combined['Host_Double_Low_Reject'] == 0)).astype(int)

eff_front_count = df_combined.groupby(['日期', '場次_num'])['is_eff_front'].transform('sum')
eff_closer_count = df_combined.groupby(['日期', '場次_num'])['is_eff_closer'].transform('sum')

df_combined['Dynamic_Pace_Bias'] = np.where(
    (eff_front_count <= 1) | (eff_closer_count == 0), 1.0,
    np.where((eff_front_count >= 3) & (eff_closer_count >= 2), -1.0, 0.0)
)
df_combined.drop(columns=['is_eff_front', 'is_eff_closer'], inplace=True, errors='ignore')

df_combined['Season_Period'] = df_combined['日期_dt'].dt.month.apply(
    lambda m: 1 if m in [9, 10, 11] else (2 if m in [12, 1, 2, 3] else 3)
)

if '配備' in df_combined.columns:
    df_combined['Gear_Str'] = df_combined['配備'].astype(str).fillna('')
    df_combined['Is_First_Time_Gear'] = df_combined['Gear_Str'].str.contains('1').astype(int)
    df_combined['Is_Remove_Gear'] = df_combined['Gear_Str'].str.contains('-').astype(int)
else:
    df_combined['Is_First_Time_Gear'] = 0
    df_combined['Is_Remove_Gear'] = 0

df_combined['獨贏賠率_num'] = pd.to_numeric(df_combined['獨贏賠率'], errors='coerce')
race_median_win = df_combined.groupby(['日期', '場次_num'])['獨贏賠率_num'].transform('median').fillna(10.0)
df_combined['獨贏賠率_num'] = df_combined['獨贏賠率_num'].fillna(race_median_win)
df_combined['獨贏賠率_num'] = np.where(df_combined['獨贏賠率_num'] <= 0, 10.0, df_combined['獨贏賠率_num'])

df_combined['位置賠率_num'] = pd.to_numeric(df_combined.get('位置賠率'), errors='coerce')
exp_pla_benchmark = 1.0 + 0.38 * (np.power(df_combined['獨贏賠率_num'], 0.72) - 1.0)
df_combined['位置賠率_num'] = df_combined['位置賠率_num'].fillna(exp_pla_benchmark)

df_combined['Expected_PLA'] = 1.0 + 0.38 * (np.power(df_combined['獨贏賠率_num'], 0.72) - 1.0)
df_combined['WIN_PLA_Ratio'] = df_combined['位置賠率_num'] / (df_combined['Expected_PLA'] + 1e-5)

df_combined['Pool_Smart_Money_Multiplier'] = np.where(
    df_combined['WIN_PLA_Ratio'] <= 0.82, 1.12,
    np.where(df_combined['WIN_PLA_Ratio'] >= 1.25, 0.88, 1.00)
)

df_combined['Implied_Prob'] = 1.0 / df_combined['獨贏賠率_num']
df_combined['Log_Odds'] = np.log1p(df_combined['獨贏賠率_num'])

bins = [0, 5.0, 10.0, 15.0, np.inf]
labels = [1, 2, 3, 4]
df_combined['Odds_Bin'] = pd.cut(df_combined['獨贏賠率_num'], bins=bins, labels=labels).astype(float).fillna(2)

df_combined['地點'] = df_combined.get('地點', 'ST').astype(str).str.strip().fillna('ST')
df_combined['路程'] = pd.to_numeric(df_combined.get('路程', 1200), errors='coerce').fillna(1200).astype(int)
df_combined['賽道設定'] = df_combined.get('賽道設定', 'A').astype(str).str.strip().fillna('A')
df_combined['檔位'] = pd.to_numeric(df_combined.get('檔位', 1), errors='coerce').fillna(1).astype(int)

if RUN_MODE == "Live_Racecard":
    df_train = df_combined[df_combined['Is_Racecard'] == 0].copy()
    df_eval = df_combined[df_combined['Is_Racecard'] == 1].copy()
else:
    df_train = df_combined[(df_combined['Is_Racecard'] == 0) & (df_combined['日期_dt'] <= '2026-05-31')].copy()
    df_eval = df_combined[(df_combined['Is_Racecard'] == 0) & (df_combined['日期_dt'] >= '2026-06-01') & (df_combined['日期_dt'] <= '2026-07-31')].copy()

top3_train = df_train[df_train['真實名次_num'].isin([1, 2, 3])].copy()

q_top3 = top3_train.groupby(['Relative_Pace_Role_Quintile', 'Dynamic_Pace_Bias'], observed=False).size()
q_tot = df_train.groupby(['Relative_Pace_Role_Quintile', 'Dynamic_Pace_Bias'], observed=False).size()
q_stats = (q_top3 / (q_tot + 1e-5)).to_dict()

draw_top3 = top3_train.groupby(['地點', '路程', '賽道設定', '檔位'], observed=False).size()
draw_tot = df_train.groupby(['地點', '路程', '賽道設定', '檔位'], observed=False).size()
draw_stats = (draw_top3 / (draw_tot + 1e-5)).to_dict()

trainer_win_rates = df_train.groupby('練馬師')['Time_Gap_Sec'].apply(lambda x: (x == 0).mean()).to_dict()
jt_win_rates = df_train.groupby(['練馬師', '騎師'])['Time_Gap_Sec'].apply(lambda x: (x == 0).mean()).to_dict()

jt_synergy_dict = {}
for (t_name, j_name), win_rate in jt_win_rates.items():
    t_rate = trainer_win_rates.get(t_name, 0.08)
    jt_synergy_dict[(str(t_name).strip(), str(j_name).strip())] = win_rate - t_rate

def apply_stats(df_target):
    df = df_target.copy()
    df['Quintile_Role_Hit_Rate'] = df.set_index(['Relative_Pace_Role_Quintile', 'Dynamic_Pace_Bias']).index.map(q_stats)
    df['Quintile_Role_Hit_Rate'] = df['Quintile_Role_Hit_Rate'].fillna(0.20)

    df['Draw_Advantage_ZScore'] = df.set_index(['地點', '路程', '賽道設定', '檔位']).index.map(draw_stats)
    df['Draw_Advantage_ZScore'] = df['Draw_Advantage_ZScore'].fillna(0.15)

    df['JT_Synergy_Delta'] = df.apply(
        lambda r: jt_synergy_dict.get((str(r.get('練馬師', '')).strip(), str(r.get('騎師', '')).strip()), 0.0), axis=1
    )
    return df

df_train = apply_stats(df_train)
df_eval = apply_stats(df_eval)

clean_train = df_train.dropna(subset=['日期_dt', '場次_num', '馬名_clean']).drop_duplicates(subset=['日期_dt', '場次_num', '馬名_clean']).copy()

for df in [clean_train, df_eval]:
    df['參賽時評分'] = pd.to_numeric(df.get('評分區域', 0), errors='coerce').fillna(0)

    # === 修正：計算最終負磅時，減去騎師的讓磅 ===
    base_weight = pd.to_numeric(df.get('實際負磅', 120), errors='coerce').fillna(120)
    allowance = pd.to_numeric(df.get('讓磅_num', 0), errors='coerce').fillna(0)
    # 取讓磅絕對值後扣除，確保例如 120 遇到 (-3) 時會變成 117
    df['實際負磅'] = base_weight - allowance.abs()

    df['馬匹體重'] = pd.to_numeric(df.get('馬匹體重', 1050), errors='coerce').fillna(1050)
    df['Load_Weight_Ratio'] = df['實際負磅'] / (df['馬匹體重'] + 1e-5)

feats_m1 = ['參賽時評分', '實際負磅', 'Load_Weight_Ratio', 'Race_Strength_Score']
feats_m2 = ['Hist_Sec_1', 'Hist_Sec_2', 'Hist_Sec_3', 'Relative_Pace_Role_Quintile', 'Pace_Role_Shift', 'Dynamic_Pace_Bias', 'Quintile_Role_Hit_Rate']
feats_m3 = ['主持A_num', '主持B_num', 'Host_Avg_Score', 'Gemini_Workload_Score', 'Host_Double_Low_Reject', 'Season_Period', 'Is_First_Time_Gear', 'Is_Remove_Gear']
feats_m4 = ['JT_Synergy_Delta', '實際負磅', '檔位', 'Draw_Advantage_ZScore']
feats_m5 = ['Odds_Bin', '獨贏賠率_num', 'Implied_Prob', 'Log_Odds', 'WIN_PLA_Ratio', '檔位']

all_feats = list(set(feats_m1 + feats_m2 + feats_m3 + feats_m4 + feats_m5))

for c in all_feats:
    clean_train[c] = pd.to_numeric(clean_train.get(c, 0), errors='coerce').fillna(0)
    df_eval[c] = pd.to_numeric(df_eval.get(c, 0), errors='coerce').fillna(0)

clean_train['Race_ID'] = clean_train['日期'].astype(str) + '_' + clean_train['場次_num'].astype(str)
gkf = GroupKFold(n_splits=5)

clean_train['S1_oof'] = 0.0
clean_train['S2_oof'] = 0.0
clean_train['S3_oof'] = 0.0
clean_train['S4_oof'] = 0.0
clean_train['S5_oof'] = 0.0

groups = clean_train['Race_ID']

print("\n=== 🔄 正在進行 5-Fold OOF 交叉驗證訓練子模型與生成 OOF 特徵 ===")
for fold, (tr_idx, va_idx) in enumerate(gkf.split(clean_train, clean_train['Target_Relevance'], groups)):
    tr_df, va_df = clean_train.iloc[tr_idx], clean_train.iloc[va_idx]

    m1_f = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(tr_df[feats_m1], tr_df['Target_Relevance'])
    m2_f = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(tr_df[feats_m2], tr_df['Target_Relevance'])
    m3_f = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(tr_df[feats_m3], tr_df['Target_Relevance'])
    m4_f = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(tr_df[feats_m4], tr_df['Target_Relevance'])
    m5_f = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(tr_df[feats_m5], tr_df['Target_Relevance'])

    clean_train.iloc[va_idx, clean_train.columns.get_loc('S1_oof')] = m1_f.predict(va_df[feats_m1])
    clean_train.iloc[va_idx, clean_train.columns.get_loc('S2_oof')] = m2_f.predict(va_df[feats_m2])
    clean_train.iloc[va_idx, clean_train.columns.get_loc('S3_oof')] = m3_f.predict(va_df[feats_m3])
    clean_train.iloc[va_idx, clean_train.columns.get_loc('S4_oof')] = m4_f.predict(va_df[feats_m4])
    clean_train.iloc[va_idx, clean_train.columns.get_loc('S5_oof')] = m5_f.predict(va_df[feats_m5])

X_oof = clean_train[['S1_oof', 'S2_oof', 'S3_oof', 'S4_oof', 'S5_oof']].values
y_true = clean_train['Target_Relevance'].values

def loss_func(weights):
    pred = X_oof @ weights
    return np.mean((pred - y_true) ** 2)

bounds = [(0.05, 0.45), (0.05, 0.45), (0.05, 0.45), (0.05, 0.45), (0.05, 0.20)]
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
init_weights = [0.20, 0.20, 0.20, 0.20, 0.20]

opt_res = minimize(loss_func, init_weights, bounds=bounds, constraints=constraints)
constrained_w = opt_res.x

X_oof_ability = clean_train[['S1_oof', 'S2_oof', 'S3_oof', 'S4_oof']].values

def loss_func_ability(weights):
    pred = X_oof_ability @ weights
    return np.mean((pred - y_true) ** 2)

bounds_ability = [(0.05, 0.50), (0.05, 0.50), (0.05, 0.50), (0.05, 0.50)]
constraints_ability = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
init_weights_ability = [0.25, 0.25, 0.25, 0.25]

opt_res_ability = minimize(loss_func_ability, init_weights_ability, bounds=bounds_ability, constraints=constraints_ability)
ability_w = opt_res_ability.x

m1 = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(clean_train[feats_m1], clean_train['Target_Relevance'])
m2 = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(clean_train[feats_m2], clean_train['Target_Relevance'])
m3 = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(clean_train[feats_m3], clean_train['Target_Relevance'])
m4 = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(clean_train[feats_m4], clean_train['Target_Relevance'])
m5 = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.03, random_state=42).fit(clean_train[feats_m5], clean_train['Target_Relevance'])

df_eval_clean = df_eval.copy()

if df_eval_clean.empty:
    print("\n==========================================================================")
    print("⚠️【執行安全中止】未發現可供預測的數據資料集 (df_eval_clean 為空 0 列)！")
    print("==========================================================================")
else:
    df_eval_clean['S1'] = m1.predict(df_eval_clean[feats_m1])
    df_eval_clean['S2'] = m2.predict(df_eval_clean[feats_m2])
    df_eval_clean['S3'] = m3.predict(df_eval_clean[feats_m3])
    df_eval_clean['S4'] = m4.predict(df_eval_clean[feats_m4])
    df_eval_clean['S5'] = m5.predict(df_eval_clean[feats_m5])

    df_eval_clean['Model_Score_Constrained'] = (
        constrained_w[0] * df_eval_clean['S1']
        + constrained_w[1] * df_eval_clean['S2']
        + constrained_w[2] * df_eval_clean['S3']
        + constrained_w[3] * df_eval_clean['S4']
        + constrained_w[4] * df_eval_clean['S5']
    )

    df_eval_clean['Model_Score_Ability'] = (
        ability_w[0] * df_eval_clean['S1']
        + ability_w[1] * df_eval_clean['S2']
        + ability_w[2] * df_eval_clean['S3']
        + ability_w[3] * df_eval_clean['S4']
    )

    df_eval_clean.loc[df_eval_clean['Host_Double_Low_Reject'] == 1, 'Model_Score_Constrained'] *= 0.5
    df_eval_clean.loc[df_eval_clean['Host_Double_Low_Reject'] == 1, 'Model_Score_Ability'] *= 0.5

    df_eval_clean['Exp_Score_Tau'] = np.exp(df_eval_clean['Model_Score_Ability'] / TEMPERATURE_TAU)
    sum_exp_tau = df_eval_clean.groupby(['日期', '場次_num'])['Exp_Score_Tau'].transform('sum')
    df_eval_clean['Model_Prob_Ability_Calibrated'] = (df_eval_clean['Exp_Score_Tau'] / (sum_exp_tau + 1e-12)) * df_eval_clean['Pool_Smart_Money_Multiplier']

    sum_prob_final = df_eval_clean.groupby(['日期', '場次_num'])['Model_Prob_Ability_Calibrated'].transform('sum')
    df_eval_clean['Model_Prob_Ability_Calibrated'] /= (sum_prob_final + 1e-12)
    df_eval_clean.drop(columns=['Exp_Score_Tau'], inplace=True)

    df_eval_clean['Edge_Value'] = df_eval_clean['Model_Prob_Ability_Calibrated'] - df_eval_clean['Implied_Prob']
    df_eval_clean['EV'] = (df_eval_clean['Model_Prob_Ability_Calibrated'] * df_eval_clean['獨贏賠率_num']) - 1.0

    df_eval_clean['Kelly_Fraction'] = np.where(
        df_eval_clean['獨贏賠率_num'] > 1.0,
        (df_eval_clean['Model_Prob_Ability_Calibrated'] * df_eval_clean['獨贏賠率_num'] - 1.0) / (df_eval_clean['獨贏賠率_num'] - 1.0),
        0.0
    )
    df_eval_clean['Kelly_Fraction'] = np.clip(df_eval_clean['Kelly_Fraction'], 0, 1.0)

    race_dfs_dict = {}

    if RUN_MODE == "Live_Racecard":
        print(f"\n==========================================================================")
        print(f"🎯【{target_date_norm} 實務賽事 AI 排位預測報告】(Version: v11.3_Production)")
        print(f"==========================================================================")

        race_groups = sorted(df_eval_clean.groupby(['日期', '場次_num']), key=lambda x: x[0][1] if pd.notna(x[0][1]) else 0)

        for (r_date, r_num), group in race_groups:
            r_num_clean = int(r_num) if pd.notna(r_num) else "未知"
            g_sorted = group.sort_values(by='Model_Score_Constrained', ascending=False)
            print(f"\n--------------------------------------------------------------------------")
            print(f"🚩 第 {r_num_clean} 場賽事預測 (參賽馬匹數: {len(g_sorted)} 匹)")
            print(f"--------------------------------------------------------------------------")

            headers = ["名次", "馬號", "馬名", "騎師", "檔位", "獨贏賠率", "位置賠率", "校準勝率", "EV期望值", "Kelly%", "多彩池/投注標籤"]
            print(f"{headers[0]:<4} {headers[1]:<4} {headers[2]:<8} {headers[3]:<6} {headers[4]:<4} {headers[5]:<6} {headers[6]:<6} {headers[7]:<8} {headers[8]:<8} {headers[9]:<7} {headers[10]}")
            print("-" * 90)

            race_records = []
            for rank_idx, (_, row) in enumerate(g_sorted.iterrows(), 1):
                h_no = str(row['馬號_clean'])
                h_name = str(row['馬名_clean'])
                jockey = str(row.get('騎師', '-'))
                trainer = str(row.get('練馬師', '-'))
                draw = str(int(row.get('檔位', 0)))
                win_odds = row['獨贏賠率_num']
                pla_odds = row['位置賠率_num']
                p_cal = row['Model_Prob_Ability_Calibrated']
                p_imp = row['Implied_Prob']
                ev = row['EV']
                kelly = row['Kelly_Fraction']
                pool_ratio = row['WIN_PLA_Ratio']

                tag = ""
                if rank_idx == 1:
                    tag += "🥇[能力首選] "
                elif rank_idx in [2, 3]:
                    tag += "🥈[主力次選] "

                if pool_ratio <= 0.82:
                    tag += "🎯[智慧資金重注] "
                elif pool_ratio >= 1.25:
                    tag += "⚠️[虛熱馬] "

                if ev > 0.15 and win_odds >= 8.0:
                    tag += "💡[精準冷門價值馬] "
                elif ev > 0.05:
                    tag += "💰[正期望值] "

                print(f" Top{rank_idx:<2} {h_no:<4} {h_name:<8} {jockey:<6} {draw:<4} {win_odds:<6.1f} {pla_odds:<6.1f} {p_cal*100:<7.2f}% {ev*100:<+7.1f}% {kelly*100:<6.1f}% {tag}")

                race_records.append({
                    '預測名次': f"Top {rank_idx}",
                    '馬號': h_no,
                    '馬名': h_name,
                    '騎師': jockey,
                    '練馬師': trainer,
                    '檔位': draw,
                    '獨贏賠率': win_odds,
                    '位置賠率': pla_odds,
                    '校準勝率': p_cal,
                    'EV期望值': ev,
                    'Kelly%': kelly,
                    'S1 (速度/近況)': row['S1'],
                    'S2 (步速/跑法)': row['S2'],
                    'S3 (狀態/配備)': row['S3'],
                    'S4 (形勢/檔位)': row['S4'],
                    'S5 (賠率/資金)': row['S5'],
                    '多彩池與投注訊號': tag.strip()
                })

            race_dfs_dict[r_num_clean] = pd.DataFrame(race_records)

    if race_dfs_dict:
        # 自動建立 Racing_Report 資料夾
        os.makedirs(EXCEL_OUTPUT_DIR, exist_ok=True)
        excel_filename = f"V11_Racing_Report_{target_date_clean_digits}.xlsx"
        excel_path = os.path.join(EXCEL_OUTPUT_DIR, excel_filename)

        # 定義高光顏色 (1~5名)
        color_fills = [
            PatternFill(start_color="FF9900", end_color="FF9900", fill_type="solid"), # 橙色 (第一名)
            PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid"), # 黃色 (第二名)
            PatternFill(start_color="92D050", end_color="92D050", fill_type="solid"), # 綠色 (第三名)
            PatternFill(start_color="00B0F0", end_color="00B0F0", fill_type="solid"), # 藍色 (第四名)
            PatternFill(start_color="7030A0", end_color="7030A0", fill_type="solid")  # 紫色 (第五名)
        ]
        s_cols = ['S1 (速度/近況)', 'S2 (步速/跑法)', 'S3 (狀態/配備)', 'S4 (形勢/檔位)', 'S5 (賠率/資金)']

        try:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                for r_num_clean, df_race in race_dfs_dict.items():
                    sheet_name = f"第 {r_num_clean} 場"
                    df_race.to_excel(writer, sheet_name=sheet_name, index=False)
                    ws = writer.sheets[sheet_name]

                    # 1. 調整欄寬與外觀設定
                    ws.views.sheetView[0].showGridLines = True
                    for col in ws.columns:
                        max_len = max(len(str(cell.value or '')) for cell in col)
                        col_letter = col[0].column_letter
                        ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

                    # 2. 設定數字格式 (百分比與小數點)
                    for row in ws.iter_rows(min_row=2, max_col=ws.max_column):
                        # 勝率, EV, Kelly% 在第 9, 10, 11 欄
                        for col_idx in [9, 10, 11]:
                            cell = row[col_idx - 1]
                            if isinstance(cell.value, (int, float)):
                                cell.number_format = '0.00%' if col_idx != 10 else '+0.0%'
                        # 獨贏與位置賠率, 以及 S1~S5 子模型分數 (第 7,8 欄 與 12~16 欄)
                        for col_idx in [7, 8, 12, 13, 14, 15, 16]:
                            cell = row[col_idx - 1]
                            if isinstance(cell.value, (int, float)):
                                cell.number_format = '0.00'

                    # 3. 為 S1~S5 分數進行自動塗色標記 (Top 5)
                    for s_col in s_cols:
                        if s_col in df_race.columns:
                            col_idx = list(df_race.columns).index(s_col) + 1
                            # 計算此模型分數的排名 (method='first' 確保名次不會因為同分而跳號，且一定有5個名額)
                            ranks = df_race[s_col].rank(method='first', ascending=False)
                            for row_i, rank in enumerate(ranks):
                                if rank <= 5:
                                    fill_idx = int(rank) - 1
                                    ws.cell(row=row_i + 2, column=col_idx).fill = color_fills[fill_idx]

            print(f"\n==========================================================================")
            print(f"📁 【V11.3 專業分頁預測報告匯出成功】")
            print(f"報表路徑: {excel_path}")
            print(f"特徵解析: 各場次已依 S1~S5 獨立列出評分，並以五色高光標記優勢馬匹。")
            print(f"==========================================================================")
        except Exception as e:
            print(f"\n⚠️ 寫入 Excel 檔案時發生錯誤: {e}")

if RUN_MODE != "Live_Racecard" and not df_eval_clean.empty:
    print(f'\n=== 📊 【Version: v11.3_Production】歷史驗證完成 ===')
    print('【全子模型下限保護 Meta-Learner 客觀學習權重】')
    print(
        f'  • S1 (速度與歷史近況): {constrained_w[0]*100:.2f}%\n'
        f'  • S2 (歷史動態步速與跑法): {constrained_w[1]*100:.2f}%\n'
        f'  • S3 (狀態、晨操與配備): {constrained_w[2]*100:.2f}%\n'
        f'  • S4 (形勢、檔位與默契): {constrained_w[3]*100:.2f}%\n'
        f'  • S5 (賠率/多彩池不比例 - 受限): {constrained_w[4]*100:.2f}%\n'
    )

Mounted at /content/drive
=== 🚀 香港賽馬五大子模型系統 [Version: v11.3_Production] ===
當前執行模式: Live_Racecard
指定預測日期: 2026-09-16

=== 📁 數據檔案讀取對接確認 ===
主特徵檔 (V7_6): /content/drive/MyDrive/HK_Racing_V7_6_Archive_Two_Hosts_Template.xlsx
沙田標準時間檔: /content/drive/MyDrive/HK_Racing/ST sectional time.csv
快活谷標準時間檔: /content/drive/MyDrive/HK_Racing/HV sectional time .csv
指定日期排位檔路徑: /content/drive/MyDrive/HK_Racing/Racecard_Stable_20260916.csv
✅ 成功載入排位檔，共 96 匹次參賽馬匹。
HKJC 動態賠率檔路徑: /content/drive/MyDrive/HKJC_Human_Reports/HKJC_Historical_Data.csv
✅ 成功載入動態賠率數據，共 5260 條紀錄。
📅 已篩選 2026-09-16 當日賠率數據，共 192 條。
🔍 識別彩池賠率欄位名稱 -> 獨贏 (WIN): [獨贏賠率] | 位置 (PLA): [位置賠率]
🔄 賠率對接結果: 成功對接 96 / 96 匹馬匹之即時獨贏賠率！

=== 🔄 正在進行 5-Fold OOF 交叉驗證訓練子模型與生成 OOF 特徵 ===

🎯【2026-09-16 實務賽事 AI 排位預測報告】(Version: v11.3_Production)

--------------------------------------------------------------------------
🚩 第 1 場賽事預測 (參賽馬匹數: 12 匹)
--------------------------------------------------------------------------
名次   馬號   馬名       騎師     檔位   獨贏賠率   位置賠率   